# 🎓 PhoBERT Depression Detection - Google Colab Training

Hướng dẫn:
1. Mở notebook này trên Google Colab
2. **Runtime > Change runtime type > GPU** (T4 hoặc V100)
3. **Mount Google Drive** để lưu model sau khi train
4. Upload các file dữ liệu cần thiết
5. Run tất cả cells theo thứ tự

In [ ]:
# ============================================================
# SECTION 1: SETUP - Cài đặt thư viện và Mount Google Drive
# ============================================================

# Mount Google Drive để lưu model
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục làm việc
import os
WORK_DIR = '/content/drive/MyDrive/phobert_training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working directory: {WORK_DIR}")
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers datasets torch accelerate scikit-learn pandas tqdm underthesea

# Kiểm tra GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ============================================================
# SECTION 2: UPLOAD DỮ LIỆU
# ============================================================

## Cách 1: Upload trực tiếp từ máy tính
Chạy cell bên dưới và chọn file từ máy tính của bạn.

In [ ]:
# Upload dữ liệu từ máy tính (chạy cell này và chọn file)
from google.colab import files

# Upload các file CSV cần thiết
print("Upload data files...")
uploaded = files.upload()

# Di chuyển file đã upload vào thư mục làm việc
for filename in uploaded.keys():
    print(f"Uploaded: {filename}")

## Cách 2: Copy từ Google Drive (nếu đã upload lên Drive trước)
Uncomment và chạy cell bên dưới nếu bạn đã upload file lên Google Drive.

In [ ]:
# Nếu data đã có trên Google Drive, uncomment dòng bên dưới
# !cp /content/drive/MyDrive/your_data_folder/*.csv /content/

# Kiểm tra các file đã có
import os
data_files = [f for f in os.listdir('.') if f.endswith('.csv')]
print(f"Data files found: {data_files}")

# ============================================================
# SECTION 3: TRAINING SCRIPT - PhoBERT Training
# ============================================================

In [ ]:
"""
PhoBERT Training Script cho Depression Detection
Chạy trực tiếp trên Google Colab với GPU
"""

from __future__ import annotations

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
)
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

# PhoBERT Model - vinai/phobert-base là pre-trained model miễn phí
# Thay = 'vinai/phubert-base-v2' nếu muốn dùng phiên bản v2
MODEL_NAME = 'vinai/phobert-base'

# Training parameters
EPOCHS = 5
BATCH_SIZE = 16          # Tăng lên 32 nếu dùng V100/A100
MAX_LEN = 128
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
SEED = 42

# Output directory
OUTPUT_DIR = Path(WORK_DIR) / 'phobert_depression_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("PhoBERT Depression Detection Training")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def set_seed(seed):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_metrics(y_true, y_pred):
    """Compute evaluation metrics."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_depression': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }

def compute_class_weights(labels):
    """Compute class weights for imbalanced dataset."""
    labels_array = np.array(labels, dtype=int)
    counts = np.bincount(labels_array, minlength=2)
    total = counts.sum()
    weights = total / (len(counts) * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float)

def normalize_text(text):
    """Simple text normalization."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

def prepare_text(text):
    """Prepare text for PhoBERT (normalize only, skip word segmentation for speed)."""
    return normalize_text(text)

In [ ]:
# ============================================================
# TEXT DATASET CLASS
# ============================================================

class TextDataset(torch.utils.data.Dataset):
    """Dataset for text classification."""
    
    def __init__(self, texts, labels, tokenizer, max_length, weights=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None
        self.weights = torch.tensor(weights, dtype=torch.float) if weights is not None else None
    
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 50)
print("LOADING DATA")
print("=" * 50)

# Đường dẫn data - điều chỉnh theo vị trí file của bạn
DATA_DIR = Path(WORK_DIR)

# Thử load từ nhiều vị trí khác nhau
possible_paths = [
    DATA_DIR / 'data' / 'final_train.csv',
    DATA_DIR / 'final_train.csv',
    Path('/content/final_train.csv'),
]

train_path = None
for p in possible_paths:
    if p.exists():
        train_path = p
        break

if train_path is None:
    # Kiểm tra thư mục hiện tại
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv') and 'train' in f.lower()]
    print(f"CSV files in current dir: {csv_files}")
    raise FileNotFoundError("Không tìm thấy file train!")

print(f"Loading from: {train_path}")

# Load data
train_df = pd.read_csv(train_path, dtype={'comment_text': str}).fillna('')
train_df['label'] = pd.to_numeric(train_df['label'], errors='coerce').astype(int)
train_df = train_df[train_df['comment_text'].str.strip().ne('')]

print(f"Train samples: {len(train_df)}")
print(f"Label distribution: {train_df['label'].value_counts().to_dict()}")

# Split train/val (80/20)
from sklearn.model_selection import train_test_split

train_texts = train_df['comment_text'].apply(prepare_text).tolist()
train_labels = train_df['label'].astype(int).tolist()

train_texts_split, val_texts, train_labels_split, val_labels = train_test_split(
    train_texts, train_labels,
    test_size=0.2,
    random_state=SEED,
    stratify=train_labels
)

print(f"Train split: {len(train_texts_split)} samples")
print(f"Validation split: {len(val_texts)} samples")

In [ ]:
# ============================================================
# LOAD TOKENIZER & MODEL
# ============================================================

print("\n" + "=" * 50)
print("LOADING PHOBERT MODEL")
print("=" * 50)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load tokenizer và model từ Hugging Face
print(f"Downloading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

print(f"Model loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# CREATE DATASETS & DATALOADERS
# ============================================================

print("\n" + "=" * 50)
print("CREATING DATASETS")
print("=" * 50)

# Create datasets
train_dataset = TextDataset(train_texts_split, train_labels_split, tokenizer, MAX_LEN)
val_dataset = TextDataset(val_texts, val_labels, tokenizer, MAX_LEN)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# ============================================================
# SETUP OPTIMIZER & SCHEDULER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = math.floor(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Class weights for imbalanced data
class_weights = compute_class_weights(train_labels_split).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Class weights: {class_weights.tolist()}")

In [ ]:
# ============================================================
# TRAINING FUNCTIONS
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, loss_fn, device, epoch):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    progress = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    
    for batch in progress:
        optimizer.zero_grad()
        
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    
    return total_loss / len(loader)

def evaluate(model, loader, device):
    """Evaluate model."""
    model.eval()
    preds = []
    labels_all = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_preds = torch.argmax(outputs.logits, dim=-1)
            
            preds.extend(batch_preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    
    return compute_metrics(labels_all, preds)

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

print("\n" + "=" * 50)
print("STARTING TRAINING")
print("=" * 50)

best_val_f1 = 0
best_model_state = None
history = []
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*50}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device, epoch)
    
    # Evaluate
    val_metrics = evaluate(model, val_loader, device)
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Validation F1 (macro): {val_metrics['f1_macro']:.4f}")
    print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
    
    # Save best model
    if val_metrics['f1_macro'] > best_val_f1:
        best_val_f1 = val_metrics['f1_macro']
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        print(f"✓ New best model! F1 = {best_val_f1:.4f}")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
    
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_f1_macro': val_metrics['f1_macro'],
        'val_accuracy': val_metrics['accuracy'],
    })
    
    # Early stopping
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\n⚠ Early stopping at epoch {epoch}")
        break

print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"Best Validation F1: {best_val_f1:.4f}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

print("\n" + "=" * 50)
print("SAVING MODEL")
print("=" * 50)

# Load best model
model.load_state_dict(best_model_state)
model.to(device)

# Save model và tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training history
results = {
    'model_name': MODEL_NAME,
    'best_val_f1': best_val_f1,
    'history': history,
    'config': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'max_len': MAX_LEN,
        'learning_rate': LEARNING_RATE,
        'train_samples': len(train_texts_split),
        'val_samples': len(val_texts),
    }
}

results_file = OUTPUT_DIR / 'training_results.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Model saved to: {OUTPUT_DIR}")
print(f"Results saved to: {results_file}")

# Verify saved files
print(f"\nSaved files:")
for f in OUTPUT_DIR.iterdir():
    print(f"  - {f.name}")

In [ ]:
# ============================================================
# DOWNLOAD MODEL VỀ MÁY (Optional)
# ============================================================

# Nếu muốn tải model về máy tính, chạy cell này
# !zip -r phobert_depression_model.zip /content/drive/MyDrive/phobert_training/phobert_depression_model/
# files.download('/content/phobert_depression_model.zip')

# Hoặc copy từ Drive về Colab local
!cp -r /content/drive/MyDrive/phobert_training/phobert_depression_model /content/
print("Model copied to /content/")

# ============================================================
# SECTION 4: HƯỚNG DẪN KẾT NỐI VỚI LOCAL PROJECT
# ============================================================

## Cách 1: Download model từ Google Drive về máy
1. Mở Google Drive của bạn
2. Tìm thư mục `phobert_training/phobert_depression_model`
3. Download về máy
4. Copy vào thư mục `models/` trong project của bạn

## Cách 2: Sử dụng Hugging Face Hub (Khuyến nghị)
1. Tạo tài khoản Hugging Face (miễn phí): https://huggingface.co/
2. Tạo new model repository
3. Upload model từ Colab:

```python
# Cài huggingface_hub
!pip install huggingface_hub

from huggingface_hub import HfApi, login

# Login với token của bạn (lấy từ https://huggingface.co/settings/tokens)
login()

# Upload model
api = HfApi()
api.upload_folder(
    folder_path='/content/drive/MyDrive/phobert_training/phobert_depression_model',
    repo_id='your_username/phobert-depression-v1',
    repo_type='model',
)
```

4. Sau đó load model bằng:
```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('your_username/phobert-depression-v1')
model = AutoModelForSequenceClassification.from_pretrained('your_username/phobert-depression-v1')
```

## Cách 3: GitHub Actions (Nâng cao)
Upload model lên GitHub repository và pull về.

# ============================================================
# SECTION 5: TEST MODEL (INFERENCE)
# ============================================================

Sau khi train xong, bạn có thể test model:

In [ ]:
# ============================================================
# INFERENCE TEST
# ============================================================

print("\n" + "=" * 50)
print("TESTING INFERENCE")
print("=" * 50)

# Load saved model
test_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=False)
test_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
test_model.to(device)
test_model.eval()

def predict(text):
    """Predict depression for a single text."""
    # Prepare text
    prepared = prepare_text(text)
    
    # Tokenize
    inputs = test_tokenizer(prepared, return_tensors='pt', truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    with torch.no_grad():
        outputs = test_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    
    return {
        'prediction': 'depression' if pred == 1 else 'normal',
        'confidence': probs[0][pred].item()
    }

# Test với vài ví dụ
test_texts = [
    "Tôi cảm thấy rất vui hôm nay, mọi thứ đều tốt đẹp!",  # normal
    "Cuộc sống này thật vô nghĩa, tôi không còn muốn sống nữa...",  # depression
    "Chán quá, không biết làm gì cả ngày hôm nay.",  # normal/mild
]

print("\nTest predictions:")
for text in test_texts:
    result = predict(text)
    print(f"\nText: {text[:50]}...")
    print(f"Prediction: {result['prediction']} (confidence: {result['confidence']:.4f})")

# ============================================================
# SECTION 6: UPGRADE LÊN COLAB PRO (Tùy chọn)
# ============================================================

## Khi nào cần Colab Pro?
- Dataset > 100,000 samples
- Cần train > 10 epochs
- Muốn dùng V100 hoặc A100
- Cần TPU thay vì GPU

## So sánh Colab Free vs Pro
| Feature | Free | Pro ($10/tháng) | Pro+ ($50/tháng) |
|---------|------|------------------|-------------------|
| GPU | T4 (shared) | V100/A100 (dedicated) | A100 (dedicated) |
| GPU Memory | ~12GB | ~40GB | ~80GB |
| Session time | ~90 min | ~24h | ~24h |
| Max runtime | 12h/day | Unlimited | Unlimited |

## Đăng ký Colab Pro
1. Truy cập https://colab.research.google.com/
2. Đăng nhập Google account
3. Click "Upgrade to Colab Pro"
4. Làm theo hướng dẫn thanh toán

## Sử dụng TPU (Tensor Processing Unit)
```python
# Chọn TPU Runtime:
# Runtime > Change runtime type > TPU

import torch_xla.core.xla_model as xm

# Chuyển model sang TPU
device = xm.xla_device()
model = model.to(device)

# TPU training cần code khác một chút
# Tham khảo: https://colab.research.google.com/github/pytorch/xla/blob/master/contrib/colab/distributed_training_tpu.ipynb
```